# Neural Latent-Rate Refinement — Interactive Notebook

Post-fit MLP-based latent rate learning anchored to mechanistic ODE priors.
Replicates `run_neural_latent_rate_refinement()` from `phoscrosstalk/neural_ode.py`.

---

**Pipeline overview:**
1. Load mechanistic fit outputs (theta_best, network arrays, data arrays).
2. Reconstruct derived-rate interpolation closures (k_act_fn, s_prod_fn).
3. Evaluate mechanistic prior trajectories at observed time points.
4. Build JAX arrays and ODE initial state.
5. Instantiate `NeuralRateGenerator` and build the neural loss closure.
6. Train with Optimistix GradientDescent (or Optax — see Section 7).
7. Visualise loss curves, fit trajectories, and learned rate functions.
8. Save all outputs to `OUTPUT_DIR` with the same file names as `neural_ode.py`.

> **The only cell you need to edit is Section 1 (User Configuration).**
> All other cells run top-to-bottom without modification.

---
## Section 0 — Imports

All imports used by `neural_ode.py`, verbatim.  
`jax.config.update("jax_enable_x64", True)` must run before any JAX array creation.

In [ ]:
from __future__ import annotations

import json
import logging
import os
import time
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib

import jax
import jax.numpy as jnp
import equinox as eqx
import diffrax
import optimistix as optx

from phoscrosstalk.config import ModelDims
from phoscrosstalk.logger import get_logger
from phoscrosstalk.mechanisms import compute_prev_site_idx, make_rhs
from phoscrosstalk.simulation import build_full_A0
from phoscrosstalk.neural_ode import (
    LatentRateMLP,
    NeuralRateGenerator,
    _make_neural_loss_fn,
    _neural_simulate_dense,
    _log_neural_loss_step,
    _neural_loss_history as neural_loss_history,
)

jax.config.update("jax_enable_x64", True)

logger = get_logger("neural_ode_notebook")

---
## Section 1 — User Configuration

**This is the only cell you need to edit.**  
Set all paths, architecture hyperparameters, training settings, loss weights,
ODE solver tolerances, and biology flags here.
Everything is bundled into a `SimpleNamespace` called `cfg` at the bottom of the cell.

In [ ]:
# ╔══════════════════════════════════════════╗
# ║         USER CONFIGURATION               ║
# ║  Edit only this cell before running all  ║
# ╚══════════════════════════════════════════╝

# ── Paths ──────────────────────────────────────────────────────────────────────
# NPZ written by main.py containing theta_best plus all problem arrays.
FITTED_PARAMS_PATH  = "results/fitted_params.npz"
# NPZ containing k_act_vals, s_prod_vals, t_obs (derived-rate snapshots).
DERIVED_RATES_PATH  = "results/derived_rates.npz"
# Directory where all neural-ODE outputs are written.
OUTPUT_DIR          = "results/neural_ode"

# ── Neural architecture ─────────────────────────────────────────────────────────
NEURAL_WIDTH  = 64    # hidden units per layer
NEURAL_DEPTH  = 3     # number of hidden layers
NEURAL_SEED   = 42    # PRNG seed for model initialisation

# ── Training ───────────────────────────────────────────────────────────────────
NEURAL_STEPS  = 2000  # total gradient steps
NEURAL_LR     = 1e-3  # learning rate
LOG_EVERY     = 100   # print/log interval (steps)

# ── Loss weights ───────────────────────────────────────────────────────────────
W_PHOSPHO           = 1.0   # phosphosite data weight
W_ABUNDANCE         = 0.5   # protein abundance data weight
W_MRNA              = 0.0   # mRNA data weight (set >0 if RNA data present)
PRIOR_WEIGHT_K_ACT  = 1.0   # regularisation toward mechanistic k_act prior
PRIOR_WEIGHT_S_PROD = 1.0   # regularisation toward mechanistic s_prod prior

# ── ODE solver ─────────────────────────────────────────────────────────────────
ODE_RTOL      = 1e-5
ODE_ATOL      = 1e-6
ODE_DT0       = 0.01
ODE_MAX_STEPS = 16384

# ── Biology ─────────────────────────────────────────────────────────────────────
RNA_RELAX     = 0.1
ABUNDANCE_MAX = 5.0
MECHANISM     = "michaelis_menten"   # must match the fitted model

# ── Dense output (smooth curves) ───────────────────────────────────────────────
SAVE_DENSE   = True    # write neural_fit_timeseries_dense.tsv
DENSE_NPOINTS = 200    # number of points in dense time grid

# ── Bundle everything into cfg ──────────────────────────────────────────────────
cfg = SimpleNamespace(
    fitted_params_path  = FITTED_PARAMS_PATH,
    derived_rates_path  = DERIVED_RATES_PATH,
    output_dir          = OUTPUT_DIR,
    # architecture
    width               = NEURAL_WIDTH,
    depth               = NEURAL_DEPTH,
    seed                = NEURAL_SEED,
    # training
    steps               = NEURAL_STEPS,
    learning_rate       = NEURAL_LR,
    print_every         = LOG_EVERY,
    # loss weights
    data_weight_phospho  = W_PHOSPHO,
    data_weight_abundance= W_ABUNDANCE,
    data_weight_mrna     = W_MRNA,
    prior_weight_k_act   = PRIOR_WEIGHT_K_ACT,
    prior_weight_s_prod  = PRIOR_WEIGHT_S_PROD,
    # ODE solver
    rtol      = ODE_RTOL,
    atol      = ODE_ATOL,
    dt0       = ODE_DT0,
    max_steps = ODE_MAX_STEPS,
    # biology
    rna_relax     = RNA_RELAX,
    abundance_max = ABUNDANCE_MAX,
    mechanism     = MECHANISM,
    # dense output
    save_dense    = SAVE_DENSE,
    dense_npoints = DENSE_NPOINTS,
)

os.makedirs(cfg.output_dir, exist_ok=True)
logger.info("[notebook] Output directory: %s", cfg.output_dir)

---
## Section 2 — Load Mechanistic Fit Outputs

Corresponds to the variable setup in `run_neural_latent_rate_refinement()`,
lines that read `problem.Cg`, `problem.Cl`, `theta_best`, `t`, `P_scaled`, etc.

We load two NPZ archives:
- **`FITTED_PARAMS_PATH`** — theta_best plus all problem arrays (Cg, Cl, K_site_kin, R, …)
- **`DERIVED_RATES_PATH`** — k_act_vals, s_prod_vals, t_obs (mechanistic prior snapshots)

Optional RNA arrays (t_rna, rna_obs_matched, rna_model_prot_idx, W_data_mrna_matched, R_data0)
are loaded gracefully with `None` fallbacks if absent.

In [ ]:
## Load mechanistic fit outputs

fp = np.load(cfg.fitted_params_path, allow_pickle=True)

# ── Core parameter vector ─────────────────────────────────────────────────────
# theta_best: shape (n_var,) float64 — best mechanistic parameter vector
theta_best = np.asarray(fp["theta_best"], dtype=np.float64)

# ── Time grid and data arrays ─────────────────────────────────────────────────
# t: shape (T,) — protein/phospho observation time points
t = np.asarray(fp["t"], dtype=np.float64)
# P_scaled: shape (N, T) — scaled phosphosite fold-change data
P_scaled = np.asarray(fp["P_scaled"], dtype=np.float64)
# A_scaled: shape (Kobs, T) — scaled protein abundance (may be shape (0, T))
A_scaled = np.asarray(fp["A_scaled"], dtype=np.float64)
# prot_idx_for_A: shape (Kobs,) int32 — maps each abundance row to a protein index
prot_idx_for_A = np.asarray(fp["prot_idx_for_A"], dtype=np.int32)
# W_data: shape (N, T) — phosphosite observation weights
W_data = np.asarray(fp["W_data"], dtype=np.float64)
# W_data_prot: shape (Kobs, T) — abundance observation weights
W_data_prot = np.asarray(fp["W_data_prot"], dtype=np.float64)

# ── Network topology arrays ───────────────────────────────────────────────────
# Cg: shape (N, N) — global crosstalk adjacency
Cg = np.asarray(fp["Cg"], dtype=np.float64)
# Cl: shape (N, N) — local (intra-protein) crosstalk adjacency
Cl = np.asarray(fp["Cl"], dtype=np.float64)
# K_site_kin: shape (N, M) — site-to-kinase assignment matrix
K_site_kin = np.asarray(fp["K_site_kin"], dtype=np.float64)
# R: shape (M, N) — kinase regulation matrix
R = np.asarray(fp["R"], dtype=np.float64)
# L_alpha: shape (M, M) — kinase-to-kinase coupling
L_alpha = np.asarray(fp["L_alpha"], dtype=np.float64)

# ── Index arrays ──────────────────────────────────────────────────────────────
# site_prot_idx: shape (N,) int32 — maps each site to its parent protein
site_prot_idx = np.asarray(fp["site_prot_idx"], dtype=np.int32)
# kin_to_prot_idx: shape (M,) int32 — maps each kinase to its protein index
kin_to_prot_idx = np.asarray(fp["kin_to_prot_idx"], dtype=np.int32)
# receptor_mask_prot: shape (K,) — boolean/float mask for receptor proteins
receptor_mask_prot = np.asarray(fp["receptor_mask_prot"], dtype=np.float64)
# receptor_mask_kin: shape (M,) — boolean/float mask for receptor kinases
receptor_mask_kin = np.asarray(fp["receptor_mask_kin"], dtype=np.float64)

# ── Scalar dimensions ─────────────────────────────────────────────────────────
# K = number of proteins, M = number of kinases, N = number of phosphosites
K = int(fp["K"]) if "K" in fp else int(receptor_mask_prot.shape[0])
M = int(fp["M"]) if "M" in fp else int(kin_to_prot_idx.shape[0])
N = int(fp["N"]) if "N" in fp else int(site_prot_idx.shape[0])

# ── Entity name lists ─────────────────────────────────────────────────────────
# proteins, sites, kinases stored as object arrays of strings
proteins = list(fp["proteins"]) if "proteins" in fp else [f"P{i}" for i in range(K)]
sites    = list(fp["sites"])    if "sites"    in fp else [f"S{i}" for i in range(N)]
kinases  = list(fp["kinases"])  if "kinases"  in fp else [f"K{i}" for i in range(M)]

# ── Optional RNA arrays ───────────────────────────────────────────────────────
# t_rna: shape (T_rna,) — mRNA observation time points (None if absent)
t_rna = np.asarray(fp["t_rna"], dtype=np.float64) if "t_rna" in fp else None
# rna_obs_matched: shape (n_rna_genes, T_rna) — scaled mRNA observations
rna_obs_matched = (
    np.asarray(fp["rna_obs_matched"], dtype=np.float64)
    if "rna_obs_matched" in fp
    else None
)
# rna_model_prot_idx: shape (n_rna_genes,) int32 — maps RNA genes to model proteins
rna_model_prot_idx = (
    np.asarray(fp["rna_model_prot_idx"], dtype=np.int32)
    if "rna_model_prot_idx" in fp
    else None
)
# W_data_mrna_matched: shape (n_rna_genes, T_rna) — mRNA observation weights
W_data_mrna_matched = (
    np.asarray(fp["W_data_mrna_matched"], dtype=np.float64)
    if "W_data_mrna_matched" in fp
    else None
)
# R_data0: shape (K, T) or (K,) — initial mRNA state from data (None if absent)
R_data0 = (
    np.asarray(fp["R_data0"], dtype=np.float64) if "R_data0" in fp else None
)

fp.close()

# ── Load derived rates ────────────────────────────────────────────────────────
dr = np.load(cfg.derived_rates_path, allow_pickle=True)
# k_act_vals: shape (K, T_obs) — mechanistic k_act at observed times
k_act_vals = np.asarray(dr["k_act_vals"], dtype=np.float64)
# s_prod_vals: shape (K, T_obs) — mechanistic s_prod at observed times
s_prod_vals = np.asarray(dr["s_prod_vals"], dtype=np.float64)
# t_obs: shape (T_obs,) — time points matching k_act_vals / s_prod_vals
t_obs = np.asarray(dr["t_obs"], dtype=np.float64)
dr.close()

# ── Register global model dimensions ─────────────────────────────────────────
ModelDims.set_dims(K, M, N)

logger.info(
    "[notebook] Loaded fit: K=%d  M=%d  N=%d  n_var=%d  t=[%.3g, %.3g]  P_scaled=%s",
    K, M, N, theta_best.shape[0], float(t[0]), float(t[-1]), str(P_scaled.shape),
)

---
## Section 3 — Reconstruct k_act_fn and s_prod_fn as Interpolation Closures

Corresponds to `run_neural_latent_rate_refinement()` lines where `k_act_fn` and `s_prod_fn`
are passed in as arguments (they are built earlier in `main.py` from `derived_rates.py`).

Here we reconstruct them from the saved `k_act_vals` / `s_prod_vals` arrays
using `scipy.interpolate.CubicSpline` (or `jnp.interp` as fallback).  
Each closure has signature `fn(t: float) -> jax.Array` of shape `(K,)`.
Output is clipped to `[0, ∞)` to guarantee non-negative rates.

In [ ]:
## Reconstruct k_act_fn and s_prod_fn as interpolation closures

try:
    from scipy.interpolate import CubicSpline

    _cs_k = CubicSpline(t_obs, k_act_vals.T, extrapolate=True)  # (T_obs, K)
    _cs_s = CubicSpline(t_obs, s_prod_vals.T, extrapolate=True)  # (T_obs, K)

    def k_act_fn(t):
        """Interpolated mechanistic k_act prior: fn(t) -> jax.Array shape (K,)."""
        t_np = float(np.asarray(t))
        vals = np.clip(_cs_k(t_np), 0.0, None)  # (K,)
        return jnp.asarray(vals, dtype=jnp.float64)

    def s_prod_fn(t):
        """Interpolated mechanistic s_prod prior: fn(t) -> jax.Array shape (K,)."""
        t_np = float(np.asarray(t))
        vals = np.clip(_cs_s(t_np), 0.0, None)  # (K,)
        return jnp.asarray(vals, dtype=jnp.float64)

    logger.info("[notebook] Derived-rate closures built using scipy.interpolate.CubicSpline.")

except ImportError:
    # Fallback: piecewise-linear interpolation with jnp.interp
    _t_obs_j = jnp.asarray(t_obs, dtype=jnp.float64)
    _k_vals_j = jnp.asarray(k_act_vals, dtype=jnp.float64)   # (K, T_obs)
    _s_vals_j = jnp.asarray(s_prod_vals, dtype=jnp.float64)  # (K, T_obs)

    def k_act_fn(t):
        """Linear-interpolation fallback: fn(t) -> jax.Array shape (K,)."""
        t_j = jnp.asarray(t, dtype=jnp.float64)
        vals = jax.vmap(lambda row: jnp.interp(t_j, _t_obs_j, row))(_k_vals_j)
        return jnp.clip(vals, 0.0, None)

    def s_prod_fn(t):
        """Linear-interpolation fallback: fn(t) -> jax.Array shape (K,)."""
        t_j = jnp.asarray(t, dtype=jnp.float64)
        vals = jax.vmap(lambda row: jnp.interp(t_j, _t_obs_j, row))(_s_vals_j)
        return jnp.clip(vals, 0.0, None)

    logger.info("[notebook] scipy unavailable — using jnp.interp fallback for rate closures.")

# Smoke-test: evaluate at t_obs[0]
_k0 = k_act_fn(float(t_obs[0]))
_s0 = s_prod_fn(float(t_obs[0]))
logger.info(
    "[notebook] k_act_fn(t0) shape=%s  s_prod_fn(t0) shape=%s",
    str(_k0.shape), str(_s0.shape),
)

---
## Section 4 — Build ODE Initial State and JAX Arrays

Corresponds to steps 2–3 of `run_neural_latent_rate_refinement()` (lines ~1232–1346):
- Evaluate mechanistic prior trajectories at observed time points.
- Build `x0` (ODE initial state) with the same nan-handling, clipping, and ordering.
- Convert all arrays to `jax.Array` with `dtype=jnp.float64` using the same variable names.

In [ ]:
## Build ODE initial state and JAX arrays

# ── Step 1: evaluate mechanistic prior trajectories at unique observed times ──
# Replicates _evaluate_mechanistic_rate_priors() call in run_neural_latent_rate_refinement
t_obs_unique = np.sort(np.unique(t)).astype(np.float64)
T_obs = len(t_obs_unique)

# k_act_init_vals / s_prod_init_vals: shape (K, T_obs) — mechanistic priors
# Use the already-loaded k_act_vals / s_prod_vals from derived rates if they align,
# else re-evaluate via the closures.
if np.allclose(t_obs_unique, t_obs):
    k_act_init_vals = k_act_vals          # (K, T_obs)
    s_prod_init_vals = s_prod_vals         # (K, T_obs)
    prior_eval_mode = "loaded from npz"
else:
    k_act_init_vals = np.stack(
        [np.asarray(k_act_fn(float(ti)), dtype=np.float64) for ti in t_obs_unique], axis=1
    )  # (K, T_obs)
    s_prod_init_vals = np.stack(
        [np.asarray(s_prod_fn(float(ti)), dtype=np.float64) for ti in t_obs_unique], axis=1
    )  # (K, T_obs)
    prior_eval_mode = "re-evaluated via closures"

logger.info(
    "[notebook] Mechanistic prior trajectories at %d observed time points (%s).",
    T_obs, prior_eval_mode,
)

# ── Step 2: build ODE initial state x0, shape (3K + M + N,) ─────────────────
# Mirrors lines 1250-1266 of run_neural_latent_rate_refinement().
T_prot = P_scaled.shape[1]
A0_full = build_full_A0(K, T_prot, A_scaled, prot_idx_for_A)  # (K, T_prot)

x0 = np.zeros(3 * K + M + N, dtype=np.float64)

# RNA initial state [0:K]
if R_data0 is not None:
    r_data = np.asarray(R_data0, dtype=np.float64)
    r0 = r_data[:, 0].copy() if r_data.ndim > 1 else r_data.copy()
    r0 = np.nan_to_num(r0, nan=1.0, posinf=5.0, neginf=0.0)
    x0[:K] = np.clip(r0, 0.0, 10.0)
else:
    x0[:K] = 1.0

# Abundance initial state [2K:3K]
a0 = np.nan_to_num(A0_full[:, 0], nan=1.0, posinf=5.0, neginf=0.0)
x0[2 * K : 3 * K] = np.clip(a0, 0.0, cfg.abundance_max)

# Phosphosite initial state [3K+M:]
p0 = np.nan_to_num(P_scaled[:, 0], nan=0.0, posinf=10.0, neginf=0.0)
x0[3 * K + M :] = np.clip(p0, 0.0, None)

# ── Step 3: build unified time grid ──────────────────────────────────────────
# Mirrors lines 1271-1284 of run_neural_latent_rate_refinement().
if t_rna is not None and len(t_rna) > 0:
    all_times = np.sort(np.union1d(t, t_rna)).astype(np.float64)
else:
    all_times = np.sort(np.unique(t)).astype(np.float64)

prot_time_idx = np.searchsorted(all_times, t)
if t_rna is not None and len(t_rna) > 0:
    mrna_time_idx = np.searchsorted(all_times, t_rna)
else:
    mrna_time_idx = None

t0_val = float(all_times[0])
t1_val = float(all_times[-1])
t_max  = float(all_times[-1]) if float(all_times[-1]) > 0 else 1.0

# ── Step 4: compute prev_site_idx ────────────────────────────────────────────
prev_site_idx = compute_prev_site_idx(site_prot_idx, N)

# ── Step 5: convert all arrays to JAX ────────────────────────────────────────
# Variable names must match neural_ode.py exactly.
theta_j             = jnp.asarray(theta_best,        dtype=jnp.float64)
y0_j                = jnp.asarray(x0,                dtype=jnp.float64)
t_eval_j            = jnp.asarray(all_times,         dtype=jnp.float64)
prot_idx_solver_j   = jnp.asarray(prot_time_idx,     dtype=jnp.int32)
prev_site_idx_j     = jnp.asarray(prev_site_idx,     dtype=jnp.int32)

Cg_j  = jnp.asarray(Cg,          dtype=jnp.float64)
Cl_j  = jnp.asarray(Cl,          dtype=jnp.float64)
K_sk_j = jnp.asarray(K_site_kin, dtype=jnp.float64)
R_j   = jnp.asarray(R,           dtype=jnp.float64)
La_j  = jnp.asarray(L_alpha,     dtype=jnp.float64)
spi_j = jnp.asarray(site_prot_idx,   dtype=jnp.int32)
k2p_j = jnp.asarray(kin_to_prot_idx, dtype=jnp.int32)
rmp_j = jnp.asarray(receptor_mask_prot, dtype=jnp.float64)
rmk_j = jnp.asarray(receptor_mask_kin,  dtype=jnp.float64)

P_data_j  = jnp.asarray(P_scaled,        dtype=jnp.float64)
A_scaled_j = jnp.asarray(A_scaled,       dtype=jnp.float64)
prot_idx_j = jnp.asarray(prot_idx_for_A, dtype=jnp.int32)
W_data_j   = jnp.asarray(W_data,         dtype=jnp.float64)
W_prot_j   = jnp.asarray(W_data_prot,    dtype=jnp.float64)

t_obs_j           = jnp.asarray(t_obs_unique,     dtype=jnp.float64)
k_act_init_obs_j  = jnp.asarray(k_act_init_vals,  dtype=jnp.float64)
s_prod_init_obs_j = jnp.asarray(s_prod_init_vals, dtype=jnp.float64)

# ── mRNA JAX arrays ───────────────────────────────────────────────────────────
has_mrna = (
    t_rna is not None
    and rna_obs_matched is not None
    and len(t_rna) > 0
    and mrna_time_idx is not None
    and rna_model_prot_idx is not None
    and len(rna_model_prot_idx) > 0
)

if has_mrna:
    rna_j          = jnp.asarray(rna_obs_matched,    dtype=jnp.float64)
    mrna_idx_j     = jnp.asarray(mrna_time_idx,      dtype=jnp.int32)
    rna_prot_idx_j = jnp.asarray(rna_model_prot_idx, dtype=jnp.int32)
    n_matched = len(rna_model_prot_idx)
    T_rna     = len(t_rna)
    W_rna_base = (
        np.asarray(W_data_mrna_matched, dtype=np.float64)
        if W_data_mrna_matched is not None
        else np.ones((n_matched, T_rna), dtype=np.float64)
    )
    W_rna_j = jnp.asarray(W_rna_base, dtype=jnp.float64)
else:
    rna_j = mrna_idx_j = rna_prot_idx_j = W_rna_j = None

logger.info(
    "[notebook] JAX arrays built: x0.shape=%s  t_eval_j.shape=%s  has_mrna=%s  t_max=%.4g",
    str(x0.shape), str(all_times.shape), str(has_mrna), t_max,
)

---
## Section 5 — Instantiate Neural Model

Corresponds to step 4 of `run_neural_latent_rate_refinement()` (lines ~1350–1518).

We use the frozen-theta path (`learn_theta=False`), which creates a `NeuralRateGenerator`
and calls `_make_neural_loss_fn()`.  
`eqx.partition(model, eqx.is_array)` separates trainable leaves (params) from static structure.

In [ ]:
## Instantiate neural model

key = jax.random.PRNGKey(int(cfg.seed))

# NeuralRateGenerator: two independent MLPs for k_hat_act(t) and s_hat_prod(t).
# Input feature vector: [normalised_time, k_act_init(t), s_prod_init(t)]  shape (1+K+K,)
neural_model = NeuralRateGenerator(
    K=K,
    width=int(cfg.width),
    depth=int(cfg.depth),
    key=key,
)

# Partition into trainable parameters and static structure.
params, static = eqx.partition(neural_model, eqx.is_array)

# Count trainable parameters.
n_params = sum(x.size for x in jax.tree_util.tree_leaves(params))

logger.info(
    "[notebook] NeuralRateGenerator created with frozen theta.  "
    "width=%d  depth=%d  seed=%d  trainable_params=%d",
    int(cfg.width), int(cfg.depth), int(cfg.seed), n_params,
)

---
## Section 6 — Build Neural Loss Function

Corresponds to step 4 of `run_neural_latent_rate_refinement()` (lines ~1468–1518).

Calls `_make_neural_loss_fn(...)` with all arguments exactly as in the script.  
Then evaluates the initial loss to confirm the closure compiled correctly.

In [ ]:
## Build neural loss function (closure)

neural_loss_fn = _make_neural_loss_fn(
    K=K,
    M=M,
    N=N,
    mechanism=cfg.mechanism,
    theta_j=theta_j,
    y0_j=y0_j,
    t0_val=t0_val,
    t1_val=t1_val,
    t_eval_j=t_eval_j,
    prot_idx_solver_j=prot_idx_solver_j,
    prev_site_idx_j=prev_site_idx_j,
    Cg_j=Cg_j,
    Cl_j=Cl_j,
    K_sk_j=K_sk_j,
    R_j=R_j,
    La_j=La_j,
    spi_j=spi_j,
    k2p_j=k2p_j,
    rmp_j=rmp_j,
    rmk_j=rmk_j,
    P_data_j=P_data_j,
    A_scaled_j=A_scaled_j,
    prot_idx_j=prot_idx_j,
    W_data_j=W_data_j,
    W_prot_j=W_prot_j,
    t_obs_j=t_obs_j,
    k_act_init_obs_j=k_act_init_obs_j,
    s_prod_init_obs_j=s_prod_init_obs_j,
    t_max=t_max,
    has_mrna=has_mrna,
    rna_j=rna_j,
    mrna_idx_j=mrna_idx_j,
    rna_prot_idx_j=rna_prot_idx_j,
    W_rna_j=W_rna_j,
    w_phospho=float(cfg.data_weight_phospho),
    w_abundance=float(cfg.data_weight_abundance),
    w_mrna=float(cfg.data_weight_mrna),
    prior_weight_k_act=float(cfg.prior_weight_k_act),
    prior_weight_s_prod=float(cfg.prior_weight_s_prod),
    rtol=float(cfg.rtol),
    atol=float(cfg.atol),
    dt0=float(cfg.dt0),
    max_steps=int(cfg.max_steps),
    rna_relax=float(cfg.rna_relax),
    abundance_max=float(cfg.abundance_max),
    k_act_init_fn=k_act_fn,
    s_prod_init_fn=s_prod_fn,
    static=static,
)

# ── Evaluate initial loss ─────────────────────────────────────────────────────
# Mirrors lines 1540-1555 of run_neural_latent_rate_refinement().
logger.info("[neural_ode] Compiling/evaluating neural loss function...")
loss_init, aux_init = neural_loss_fn(params, None)
aux_vals = [float(x) for x in tuple(aux_init)]
while len(aux_vals) < 7:
    aux_vals.append(0.0)
f_p0, f_a0, f_r0, f_k0, f_s0, f_theta0, f_traj0 = aux_vals[:7]

logger.info(
    "[neural_ode] Initial loss: total=%.4g | phospho=%.4g | abund=%.4g "
    "| mrna=%.4g | k_prior=%.4g | s_prior=%.4g | theta_prior=%.4g | traj_prior=%.4g",
    float(loss_init), f_p0, f_a0, f_r0, f_k0, f_s0, f_theta0, f_traj0,
)

---
## Section 7 — Train Neural Rate Generator

Corresponds to step 7 of `run_neural_latent_rate_refinement()` (lines ~1584–1620),
Optimistix GradientDescent path (`use_optax=False`).

Because `optx.minimise` runs a JIT-compiled while_loop, per-step loss is captured
via `jax.debug.callback` into `neural_loss_history` (populated by `_log_neural_loss_step`).

The list is cleared before training so it only contains the current run.

In [ ]:
## Train neural rate generator

# Clear the module-level loss history before starting.
neural_loss_history.clear()

logger.info(
    "[neural_ode] Training with Optimistix GradientDescent for %d steps.",
    int(cfg.steps),
)

lr = float(cfg.learning_rate)
solver = optx.GradientDescent(
    learning_rate=lr,
    rtol=1e-100,
    atol=1e-100,
)

t_total = time.perf_counter()
sol = optx.minimise(
    neural_loss_fn,
    solver,
    params,
    args=None,
    has_aux=True,
    max_steps=int(cfg.steps),
    throw=False,
)

# Block until all JAX computations are done.
for leaf in jax.tree_util.tree_leaves(sol.value):
    if hasattr(leaf, "block_until_ready"):
        leaf.block_until_ready()
jax.effects_barrier()

total_elapsed = time.perf_counter() - t_total
params_opt = sol.value
train_result_status = str(sol.result)

logger.info(
    "[neural_ode] refinement complete result=%s total_t=%.1fs",
    train_result_status, total_elapsed,
)

# ── Final loss ────────────────────────────────────────────────────────────────
loss_final, aux_final = neural_loss_fn(params_opt, None)
aux_final_vals = [float(x) for x in tuple(aux_final)]
while len(aux_final_vals) < 7:
    aux_final_vals.append(0.0)
f_p_fin, f_a_fin, f_r_fin, f_k_fin, f_s_fin, f_theta_fin, f_traj_fin = aux_final_vals[:7]

logger.info(
    "[neural_ode] Final loss: total=%.4g | phospho=%.4g | abund=%.4g "
    "| mrna=%.4g | k_prior=%.4g | s_prior=%.4g | theta_prior=%.4g | traj_prior=%.4g",
    float(loss_final), f_p_fin, f_a_fin, f_r_fin, f_k_fin, f_s_fin, f_theta_fin, f_traj_fin,
)

---
## Section 8 — Loss Curve Visualisation

Corresponds to step 13 of `run_neural_latent_rate_refinement()` (training losses TSV, lines ~1860–1892).

Builds a `pandas.DataFrame` from `neural_loss_history`, plots total and component losses
on log-scale axes, and saves to `OUTPUT_DIR/neural_loss_curve.png`.

In [ ]:
## Loss curve

# Build DataFrame from the loss history collected during training.
if neural_loss_history:
    df_losses = pd.DataFrame(neural_loss_history)
else:
    # Fallback if no per-step data was captured (Optimistix path without debug.callback).
    df_losses = pd.DataFrame([
        {
            "step": 0,
            "neural_loss_total":      float(loss_init),
            "neural_loss_phospho":    f_p0,
            "neural_loss_abundance":  f_a0,
            "neural_loss_mrna":       f_r0,
            "neural_loss_k_act_prior": f_k0,
            "neural_loss_s_prod_prior": f_s0,
            "neural_loss_theta_prior": f_theta0,
            "neural_loss_traj_prior":  f_traj0,
        },
        {
            "step": int(cfg.steps),
            "neural_loss_total":      float(loss_final),
            "neural_loss_phospho":    f_p_fin,
            "neural_loss_abundance":  f_a_fin,
            "neural_loss_mrna":       f_r_fin,
            "neural_loss_k_act_prior": f_k_fin,
            "neural_loss_s_prod_prior": f_s_fin,
            "neural_loss_theta_prior": f_theta_fin,
            "neural_loss_traj_prior":  f_traj_fin,
        },
    ])

# Save losses TSV (same as run_neural_latent_rate_refinement).
losses_path = os.path.join(cfg.output_dir, "neural_training_losses.tsv")
df_losses.to_csv(losses_path, sep="\t", index=False)
logger.info("[neural_ode] Saved %s", losses_path)

# ── Plot ──────────────────────────────────────────────────────────────────────
component_cols = [
    "neural_loss_phospho",
    "neural_loss_abundance",
    "neural_loss_mrna",
    "neural_loss_k_act_prior",
    "neural_loss_s_prod_prior",
]
component_labels = ["phospho", "abundance", "mrna", "k_act prior", "s_prod prior"]

steps_col = df_losses["step"] if "step" in df_losses.columns else df_losses.index

fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
fig.suptitle("Neural Latent-Rate Refinement — Training Loss", fontsize=13)

# Top: total loss
axes[0].plot(steps_col, df_losses["neural_loss_total"], color="steelblue", lw=2, label="total")
axes[0].set_yscale("log")
axes[0].set_ylabel("Total loss (log scale)")
axes[0].legend(frameon=False)
axes[0].grid(True, which="both", alpha=0.3)

# Bottom: component losses
colors = plt.cm.tab10.colors
for col, label, color in zip(component_cols, component_labels, colors):
    if col in df_losses.columns:
        axes[1].plot(steps_col, df_losses[col], label=label, color=color, lw=1.5)
axes[1].set_yscale("log")
axes[1].set_ylabel("Component loss (log scale)")
axes[1].set_xlabel("Training step")
axes[1].legend(frameon=False, ncol=2, fontsize=9)
axes[1].grid(True, which="both", alpha=0.3)

plt.tight_layout()
loss_curve_path = os.path.join(cfg.output_dir, "neural_loss_curve.png")
fig.savefig(loss_curve_path, dpi=150, bbox_inches="tight")
plt.show()
logger.info("[neural_ode] Saved %s", loss_curve_path)

---
## Section 9 — Reconstruct Optimal Model and Simulate

Corresponds to steps 8 and 11–12 of `run_neural_latent_rate_refinement()` (lines ~1643–1855).

- Reconstruct `neural_model_opt` from trained params and static structure.
- Simulate at observed time points and save `neural_fit_timeseries.tsv`.
- Optionally simulate at a dense grid and save `neural_fit_timeseries_dense.tsv`.

In [ ]:
## Reconstruct optimal neural model

neural_model_opt = eqx.combine(params_opt, static)

# For frozen-theta path, theta_for_outputs_j is the original theta_j.
theta_for_outputs_j = theta_j

logger.info("[neural_ode] Optimal neural model reconstructed from trained params.")

In [ ]:
## Simulate at observed time points

# Mirrors lines 1731-1790 of run_neural_latent_rate_refinement().
sim_obs = _neural_simulate_dense(
    model=neural_model_opt,
    K=K,
    M=M,
    N=N,
    mechanism=cfg.mechanism,
    theta_j=theta_for_outputs_j,
    y0_j=y0_j,
    t0_val=t0_val,
    t_dense=t_obs_unique,
    prev_site_idx_j=prev_site_idx_j,
    Cg_j=Cg_j,
    Cl_j=Cl_j,
    K_sk_j=K_sk_j,
    R_j=R_j,
    La_j=La_j,
    spi_j=spi_j,
    k2p_j=k2p_j,
    rmp_j=rmp_j,
    rmk_j=rmk_j,
    t_max=t_max,
    k_act_init_fn=k_act_fn,
    s_prod_init_fn=s_prod_fn,
    rtol=float(cfg.rtol),
    atol=float(cfg.atol),
    dt0=float(cfg.dt0),
    max_steps=int(cfg.max_steps),
    rna_relax=float(cfg.rna_relax),
    abundance_max=float(cfg.abundance_max),
)

# Build timeseries DataFrame: same structure as run_neural_latent_rate_refinement().
ts_rows = []
for ti_idx, t_val in enumerate(t_obs_unique):
    for s_idx, site in enumerate(sites):
        ts_rows.append({
            "time":           float(t_val),
            "entity_type":    "phosphosite",
            "entity":         site,
            "value_neural":   float(sim_obs["P_sim"][s_idx, ti_idx]),
            "value_observed": float(P_scaled[s_idx, ti_idx]) if ti_idx < P_scaled.shape[1] else float("nan"),
        })
    for p_idx, prot in enumerate(proteins):
        ts_rows.append({
            "time":           float(t_val),
            "entity_type":    "abundance",
            "entity":         prot,
            "value_neural":   float(sim_obs["A_sim"][p_idx, ti_idx]),
            "value_observed": float("nan"),
        })

df_ts = pd.DataFrame(ts_rows)
ts_path = os.path.join(cfg.output_dir, "neural_fit_timeseries.tsv")
df_ts.to_csv(ts_path, sep="\t", index=False)
logger.info("[neural_ode] Saved %s", ts_path)
df_ts.head()

In [ ]:
## Dense simulation (smooth curves)

# Mirrors lines 1795-1855 of run_neural_latent_rate_refinement().
if cfg.save_dense:
    n_dense = int(cfg.dense_npoints)
    t_dense_arr = np.linspace(float(t_obs_unique[0]), float(t_obs_unique[-1]), n_dense)

    sim_dense = _neural_simulate_dense(
        model=neural_model_opt,
        K=K,
        M=M,
        N=N,
        mechanism=cfg.mechanism,
        theta_j=theta_for_outputs_j,
        y0_j=y0_j,
        t0_val=t0_val,
        t_dense=t_dense_arr,
        prev_site_idx_j=prev_site_idx_j,
        Cg_j=Cg_j,
        Cl_j=Cl_j,
        K_sk_j=K_sk_j,
        R_j=R_j,
        La_j=La_j,
        spi_j=spi_j,
        k2p_j=k2p_j,
        rmp_j=rmp_j,
        rmk_j=rmk_j,
        t_max=t_max,
        k_act_init_fn=k_act_fn,
        s_prod_init_fn=s_prod_fn,
        rtol=float(cfg.rtol),
        atol=float(cfg.atol),
        dt0=float(cfg.dt0),
        max_steps=int(cfg.max_steps),
        rna_relax=float(cfg.rna_relax),
        abundance_max=float(cfg.abundance_max),
    )

    dense_rows = []
    for ti_idx, t_val in enumerate(t_dense_arr):
        for s_idx, site in enumerate(sites):
            dense_rows.append({
                "time":        float(t_val),
                "entity_type": "phosphosite",
                "entity":      site,
                "value":       float(sim_dense["P_sim"][s_idx, ti_idx]),
                "series_type": "neural_refined_dense",
            })
        for p_idx, prot in enumerate(proteins):
            dense_rows.append({
                "time":        float(t_val),
                "entity_type": "abundance",
                "entity":      prot,
                "value":       float(sim_dense["A_sim"][p_idx, ti_idx]),
                "series_type": "neural_refined_dense",
            })

    df_dense = pd.DataFrame(dense_rows)
    dense_path = os.path.join(cfg.output_dir, "neural_fit_timeseries_dense.tsv")
    df_dense.to_csv(dense_path, sep="\t", index=False)
    logger.info("[neural_ode] Saved %s", dense_path)
else:
    logger.info("[neural_ode] save_dense=False — skipping dense simulation.")

---
## Section 10 — Learned vs Mechanistic Rate Trajectories

Corresponds to steps 9–10 of `run_neural_latent_rate_refinement()` (lines ~1665–1726).

Evaluates learned `k_hat(t)` and `s_hat(t)` from `neural_model_opt` at each observed time,
compares to mechanistic priors, saves a TSV, and plots per-protein trajectories.

In [ ]:
## Inspect learned vs mechanistic rates

# Build feature matrix for all observed times (mirrors lines 1668-1678).
t_norms_obs = np.clip(t_obs_unique / t_max, 0.0, 1.0)[:, None]   # (T_obs, 1)
k_priors_obs_np = k_act_init_vals.T                                # (T_obs, K)
s_priors_obs_np = s_prod_init_vals.T                               # (T_obs, K)
features_obs_np = np.concatenate([t_norms_obs, k_priors_obs_np, s_priors_obs_np], axis=1)
features_obs_j  = jnp.asarray(features_obs_np, dtype=jnp.float64)

k_hats_obs_j, s_hats_obs_j = jax.vmap(neural_model_opt)(features_obs_j)
k_hats_obs = np.asarray(k_hats_obs_j)  # (T_obs, K)
s_hats_obs = np.asarray(s_hats_obs_j)  # (T_obs, K)

# ── Save learned-rates TSV ────────────────────────────────────────────────────
# Mirrors lines 1683-1715 of run_neural_latent_rate_refinement().
rate_rows = []
for ti_idx, t_val in enumerate(t_obs_unique):
    for p_idx, prot in enumerate(proteins):
        rate_rows.append({
            "time":             float(t_val),
            "protein":          prot,
            "k_hat":            float(k_hats_obs[ti_idx, p_idx]),
            "k_act_prior":      float(k_act_init_vals[p_idx, ti_idx]),
            "s_hat":            float(s_hats_obs[ti_idx, p_idx]),
            "s_prod_prior":     float(s_prod_init_vals[p_idx, ti_idx]),
        })

df_rates = pd.DataFrame(rate_rows)
rates_tsv = os.path.join(cfg.output_dir, "neural_learned_rates.tsv")
df_rates.to_csv(rates_tsv, sep="\t", index=False)
logger.info("[neural_ode] Saved %s", rates_tsv)

# Also save the latent rates in the format matching neural_ode.py.
np.savez(
    os.path.join(cfg.output_dir, "neural_latent_rates.npz"),
    t_obs=t_obs_unique,
    proteins=np.array(proteins),
    k_act_mechanistic=k_act_init_vals,
    s_prod_mechanistic=s_prod_init_vals,
    k_act_neural=k_hats_obs.T,
    s_prod_neural=s_hats_obs.T,
)

# ── Plot per-protein rate trajectories ───────────────────────────────────────
n_cols = min(4, K)
n_rows = max(1, (K + n_cols - 1) // n_cols)
fig, axes = plt.subplots(n_rows, n_cols * 2, figsize=(4 * n_cols * 2, 3 * n_rows))
axes = np.array(axes).reshape(n_rows, n_cols * 2)

for p_idx, prot in enumerate(proteins):
    row = p_idx // n_cols
    col_k = (p_idx % n_cols) * 2
    col_s = col_k + 1

    ax_k = axes[row, col_k]
    ax_s = axes[row, col_s]

    # k_act
    ax_k.plot(t_obs_unique, k_hats_obs[:, p_idx], lw=2, label="k_hat (neural)", color="steelblue")
    ax_k.plot(t_obs_unique, k_act_init_vals[p_idx, :], lw=1.5, ls="--", label="k_act (mech.)", color="gray")
    ax_k.set_title(f"{prot} — k_act", fontsize=9)
    ax_k.set_xlabel("time")
    ax_k.legend(fontsize=7, frameon=False)

    # s_prod
    ax_s.plot(t_obs_unique, s_hats_obs[:, p_idx], lw=2, label="s_hat (neural)", color="darkorange")
    ax_s.plot(t_obs_unique, s_prod_init_vals[p_idx, :], lw=1.5, ls="--", label="s_prod (mech.)", color="gray")
    ax_s.set_title(f"{prot} — s_prod", fontsize=9)
    ax_s.set_xlabel("time")
    ax_s.legend(fontsize=7, frameon=False)

# Hide unused subplots.
for idx in range(K, n_rows * n_cols):
    row = idx // n_cols
    for c in [idx % n_cols * 2, idx % n_cols * 2 + 1]:
        if c < axes.shape[1]:
            axes[row, c].set_visible(False)

fig.suptitle("Learned vs Mechanistic Rate Trajectories", fontsize=13)
plt.tight_layout()
rates_png = os.path.join(cfg.output_dir, "neural_learned_rates.png")
fig.savefig(rates_png, dpi=150, bbox_inches="tight")
plt.show()
logger.info("[neural_ode] Saved %s", rates_png)

---
## Section 11 — Save Metadata JSON

Corresponds to step 14 of `run_neural_latent_rate_refinement()` (lines ~1897–1966).

Saves `neural_metadata.json` to `OUTPUT_DIR` with exactly the same keys as the script.

In [ ]:
## Save metadata JSON

metadata = {
    "enabled":       True,
    "width":         int(cfg.width),
    "depth":         int(cfg.depth),
    "steps":         int(cfg.steps),
    "seed":          int(cfg.seed),
    "optimizer":     "optimistix.GradientDescent",
    "learning_rate": float(cfg.learning_rate),
    "rtol":          float(cfg.rtol),
    "atol":          float(cfg.atol),
    "dt0":           float(cfg.dt0),
    "max_steps":     int(cfg.max_steps),
    "prior_weights": {
        "kact":  float(cfg.prior_weight_k_act),
        "sprod": float(cfg.prior_weight_s_prod),
    },
    "data_weights": {
        "phospho":   float(cfg.data_weight_phospho),
        "abundance": float(cfg.data_weight_abundance),
        "mrna":      float(cfg.data_weight_mrna),
    },
    "theta_fixed":     True,
    "state_variables": ["R", "S", "A", "Kdyn", "P"],
    "initial_loss":    float(loss_init),
    "final_loss":      float(loss_final),
    "note": (
        "Post-fit latent-rate refinement. theta_best is held fixed. "
        "Neural rates are regularised toward mechanistic priors."
    ),
}

meta_path = os.path.join(cfg.output_dir, "neural_metadata.json")
with open(meta_path, "w") as fh:
    json.dump(metadata, fh, indent=2)
logger.info("[neural_ode] Saved %s", meta_path)

---
## Section 12 — Save Trained Neural Model

Saves the trained parameter leaves to `OUTPUT_DIR/neural_model_params.npz` using
`eqx.tree_serialise_leaves` (preferred) or manual extraction via `jax.tree_util`.

Corresponds to the model-saving pattern in `run_neural_latent_rate_refinement()`;  
the serialised file can be loaded later for inference without re-training.

In [ ]:
## Save trained neural model

model_save_path = os.path.join(cfg.output_dir, "neural_model_params.eqx")

try:
    # Preferred: eqx.tree_serialise_leaves writes a binary checkpoint.
    eqx.tree_serialise_leaves(model_save_path, params_opt)
    logger.info("[neural_ode] Saved trained model params (eqx): %s", model_save_path)
except Exception as _err:
    logger.warning(
        "[neural_ode] eqx.tree_serialise_leaves failed (%s) — falling back to npz.",
        _err,
    )
    # Fallback: extract leaves manually and save to npz.
    leaves = jax.tree_util.tree_leaves(params_opt)
    npz_path = os.path.join(cfg.output_dir, "neural_model_params.npz")
    np.savez(
        npz_path,
        **{f"leaf_{i:04d}": np.asarray(leaf, dtype=np.float64) for i, leaf in enumerate(leaves)},
    )
    model_save_path = npz_path
    logger.info("[neural_ode] Saved trained model params (npz): %s", model_save_path)

print(f"Model saved to: {model_save_path}")